# Customer Churn Prediction with XGBoost


In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer

import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker/1-processing" # importante aqui hacer referencia a donde se encuentran los csv en S3

# Define IAM role
import boto3
from sagemaker import get_execution_role

role = get_execution_role()

Se elimina del notebook original la celda que hacía que fuera necesario subir al entorno del jupyter los csv. Ahora los coge directamente de S3 habiendo sido generados anteriormente con el Job de Processing

---
## Train

First we'll need to specify the locations of the XGBoost algorithm containers.

In [ ]:
container = sagemaker.image_uris.retrieve("xgboost", sess.boto_region_name, "1.7-1")
display(container)

Then, we'll create `TrainingInput` so that our training function can use as a pointer to the files in S3.

In [ ]:
s3_input_train = TrainingInput(s3_data=f"s3://{bucket}/{prefix}/train", content_type="csv")
s3_input_validation = TrainingInput(s3_data=f"s3://{bucket}/{prefix}/validation/", content_type="csv")

Now, we can specify a few parameters like what type of training instances we'd like to use and how many, as well as our XGBoost hyperparameters.

The complete list of hyperparameters can be found in the [documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost_hyperparameters.html).

In [ ]:
sess = sagemaker.Session()

xgb = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=sess,
)
xgb.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=0,
    objective="binary:logistic",
    eval_metric="auc",
    early_stopping_rounds=10,
    num_round=100,
)

xgb.fit({"train": s3_input_train, "validation": s3_input_validation})

---
## Deploy

Now that we've trained the algorithm, let's create a model and deploy it to a hosted endpoint.

In [ ]:
xgb_predictor = xgb.deploy(
    initial_instance_count=1, instance_type="ml.m5.xlarge", serializer=CSVSerializer()
)

### Evaluate

Now that we have a hosted endpoint running, we can make real-time predictions from our model very easily, simply by making a `http` POST request.  But first, we'll need to set up serializers and deserializers for passing our `test_data` NumPy arrays to the model behind the endpoint.


Se cambia esta celda para que el test lo coja directamente del S3 generado por el Job Processing

In [ ]:
test_data = pd.read_csv(f"s3://{bucket}/{prefix}/test/test.csv")

Now, we'll use a simple function to:
1. Loop over our test dataset
1. Split it into mini-batches of rows 
1. Convert those mini-batchs to CSV string payloads
1. Retrieve mini-batch predictions by invoking the XGBoost endpoint
1. Collect predictions and convert from the CSV output our model provides into a NumPy array

In [ ]:
def predict(data, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = ""
    for array in split_array:
        predictions = "".join([predictions, xgb_predictor.predict(array).decode("utf-8")])

    return predictions.split("\n")[:-1]


predictions = predict(test_data.to_numpy()[:, 1:])

In [ ]:
predictions = np.array([float(num) for num in predictions])
print(predictions)

There are many ways to compare the performance of a machine learning model, but let's start by simply by comparing actual to predicted values.  In this case, we're simply predicting whether the customer churned (`1`) or not (`0`), which produces a confusion matrix.

In [ ]:
pd.crosstab(
    index=test_data.iloc[:, 0],
    columns=np.round(predictions),
    rownames=["actual"],
    colnames=["predictions"],
)

_Note, due to randomized elements of the algorithm, your results may differ slightly._

---
## Clean-up to avoid excessive costs

If you're ready to be done with this notebook, please run the cell below.  This will remove the hosted endpoint you created and avoid any charges from a stray instance being left on.

In [ ]:
xgb_predictor.delete_endpoint()